[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/3_Multimodal.ipynb)

**Día 1 · Cuaderno 3 de 3** · Acompaña al tema 3 de la web: [Modelos multimodales](https://datamecum.sergiovillanuevalopez.com/temas/03-multimodalidad/)

**En clase:** hasta *Cuándo usar cada uno*. **Extra**, si da tiempo: un prompt para cada modelo. **Para casa:** más cosas con Qwen y CLIP, y una imagen tuya.

# Modelos multimodales: visión y lenguaje

Los modelos multimodales entienden a la vez imágenes y texto. Eso permite pedirles cosas en lenguaje natural, sin una lista cerrada de clases y sin entrenar nada. Vemos cuatro, de menos a más capaz:

| Modelo | Qué hace |
|---|---|
| **CLIP** | Dice cuál de varios textos describe mejor una imagen (clasificación *zero-shot*) |
| **BLIP** | Describe una imagen con una frase |
| **Grounding DINO** | Detecta los objetos que le nombras en un texto |
| **Qwen3.5** | Un modelo de visión y lenguaje (VLM): responde preguntas, lee texto, razona y localiza objetos siguiendo instrucciones |

CLIP, BLIP y Grounding DINO esperan el texto **en inglés**, que es el idioma en el que se entrenaron. Qwen entiende también el castellano.

**Antes de empezar**

1. Activa la GPU: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.
2. Ejecuta las celdas en orden, de arriba abajo, con `Mayúsculas + Intro`.
3. No hay que instalar nada: todas las librerías que usamos ya vienen en Colab. Los modelos se descargan solos desde Hugging Face la primera vez que se usan.

Este cuaderno es tuyo: al abrirlo desde el enlace se crea una copia que solo ves tú, y puedes cambiar lo que quieras sin estropear el original.

## Preparación

In [ ]:
import json
import torch
import requests
import transformers
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
# En GPU usamos media precisión (float16): ocupa la mitad de memoria y va más rápido.
dtype = torch.float16 if device == "cuda" else torch.float32
print("Usando:", device)
transformers.logging.set_verbosity_error()   # oculta avisos internos que no afectan al resultado

ASSETS = "https://raw.githubusercontent.com/sergiovillanueva/Modelos_Fundacionales/main/assets/"


def cargar_imagen(nombre_o_url):
    """Abre una imagen del curso en RGB.

    Si el cuaderno se ejecuta dentro de la carpeta del curso descargada (por ejemplo, sin internet),
    la toma de assets/; si no, la descarga del repositorio del curso o de la URL indicada.
    """
    local = Path("assets") / Path(nombre_o_url).name
    if local.exists():
        return Image.open(local).convert("RGB")
    url = nombre_o_url if nombre_o_url.startswith("http") else ASSETS + nombre_o_url
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content)).convert("RGB")


def mostrar(imagen, titulo=""):
    """Enseña una imagen con un título."""
    plt.imshow(imagen)
    plt.title(titulo)
    plt.axis("off")
    plt.show()


image_cars = cargar_imagen("cars.jpg")
image_person_cars = cargar_imagen("person_cars.jpg")
image_fruits = cargar_imagen("fruits.jpg")
image_board = cargar_imagen("board.jpg")
image_park = cargar_imagen("park.jpg")
image_room = cargar_imagen("000000000139.jpg")
image_carpet_ok = cargar_imagen("carpet_ok.jpg")
image_carpet_nok = cargar_imagen("carpet_nok.jpg")
print("Imágenes cargadas")

## CLIP: clasificar sin entrenar

CLIP se entrenó con 400 millones de pares de imagen y texto sacados de internet. Aprendió a poner en el mismo espacio la imagen de un perro y la frase "a photo of a dog", de modo que se puede medir cuánto se parecen.

Para clasificar le damos una imagen y varias descripciones, y nos dice cuál encaja mejor. Las clases las inventas tú en cada llamada: por eso se llama *zero-shot*, sin ejemplos de entrenamiento.

Paper: [Radford et al., 2021](https://arxiv.org/abs/2103.00020)

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
print("CLIP cargado")

In [ ]:
def clasificar_con_clip(imagen, textos):
    """Muestra la imagen y la probabilidad de cada texto según CLIP, con un gráfico de barras."""
    entradas = clip_processor(text=textos, images=imagen, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        probabilidades = clip_model(**entradas).logits_per_image.softmax(dim=1)[0]

    figura, ejes = plt.subplots(1, 2, figsize=(14, 4))
    ejes[0].imshow(imagen)
    ejes[0].axis("off")
    ejes[1].barh(textos, probabilidades.cpu(), color="#2727c7")
    ejes[1].invert_yaxis()   # el primer texto, arriba
    ejes[1].set_xlim(0, 1)
    ejes[1].set_xlabel("probabilidad")
    plt.tight_layout()
    plt.show()


clasificar_con_clip(image_cars, ["a parking lot with cars", "a factory", "an office", "a park"])

Las clases son las que quieras. Prueba a cambiar los textos:

In [ ]:
clasificar_con_clip(image_fruits, ["a photo of fruit", "a photo of cars", "a photo of a dog"])
# clasificar_con_clip(image_carpet_nok, ["a clean carpet", "a carpet with a defect"])
# clasificar_con_clip(image_board, ["an electronic circuit board", "a kitchen", "a car engine"])

## BLIP: describir una imagen

BLIP genera una frase que describe la imagen. Sirve para crear descripciones automáticas, indexar un archivo de fotos o hacer accesibles las imágenes para personas ciegas.

Paper: [Li et al., 2022](https://arxiv.org/abs/2201.12086)

In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

blip_name = "Salesforce/blip-image-captioning-base"
blip_processor = BlipProcessor.from_pretrained(blip_name)
blip_model = BlipForConditionalGeneration.from_pretrained(blip_name, dtype=dtype).to(device)
print("BLIP cargado")

In [ ]:
def describir(imagen):
    """Enseña la imagen con la frase que genera BLIP como título."""
    entradas = blip_processor(images=imagen, return_tensors="pt").to(device, dtype)
    with torch.no_grad():
        salida = blip_model.generate(**entradas, max_new_tokens=40)
    mostrar(imagen, blip_processor.decode(salida[0], skip_special_tokens=True))


describir(image_person_cars)
describir(image_board)

## Grounding DINO: detectar lo que nombras

Grounding DINO es un detector como el del cuaderno 1, pero en vez de 80 clases fijas detecta lo que le escribas: "kiwi", "yellow car", "person with a red shirt".

El texto tiene un formato concreto: **en minúsculas y cada objeto terminado en punto**, por ejemplo `"kiwi. apple."`.

Ojo con el nombre: este DINO es un detector y no tiene nada que ver con el DINO autosupervisado del cuaderno 4.

Paper: [Liu et al., 2023](https://arxiv.org/abs/2303.05499)

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

dino_name = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(dino_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_name).to(device)
print("Grounding DINO cargado")

In [ ]:
def dibujar_cajas(imagen, cajas, etiquetas, titulo=""):
    """Dibuja cada caja [x1, y1, x2, y2] sobre la imagen, con su etiqueta encima."""
    plt.figure(figsize=(12, 8))
    plt.imshow(imagen)
    for (x1, y1, x2, y2), etiqueta in zip(cajas, etiquetas):
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", linewidth=3))
        plt.text(x1, y1, etiqueta, color="white", backgroundcolor="red")
    plt.title(titulo)
    plt.axis("off")
    plt.show()


def detectar_por_texto(imagen, texto, umbral=0.35):
    """Detecta y dibuja los objetos nombrados en el texto, por ejemplo "kiwi. apple."."""
    entradas = dino_processor(images=imagen, text=texto, return_tensors="pt").to(device)
    with torch.no_grad():
        salidas = dino_model(**entradas)
    resultado = dino_processor.post_process_grounded_object_detection(
        salidas, entradas.input_ids, threshold=umbral, text_threshold=0.25, target_sizes=[imagen.size[::-1]]
    )[0]

    etiquetas = [f"{nombre} {puntuacion:.2f}" for nombre, puntuacion in zip(resultado["labels"], resultado["scores"])]
    dibujar_cajas(imagen, resultado["boxes"].tolist(), etiquetas, texto)


detectar_por_texto(image_fruits, "kiwi. apple.")
detectar_por_texto(image_person_cars, "yellow car. woman.")

## Qwen3.5: un modelo para casi todo

Hasta aquí cada modelo hacía una cosa: CLIP clasifica, BLIP describe y Grounding DINO detecta. Un VLM (*Vision-Language Model*) es un modelo de lenguaje, como los de los chats, al que se le ha añadido la vista. Se le habla como a un chat: responde preguntas sobre la imagen, lee el texto que aparece en ella, razona sobre lo que ve y localiza objetos.

Usamos **Qwen3.5-2B**, de Alibaba (2026, licencia Apache 2.0). Es de los VLM más pequeños que funcionan bien, pero son 4,5 GB de descarga: **la primera carga tarda unos minutos**.

In [ ]:
from transformers import AutoModelForImageTextToText

vlm_name = "Qwen/Qwen3.5-2B"
vlm_processor = AutoProcessor.from_pretrained(vlm_name)
vlm_model = AutoModelForImageTextToText.from_pretrained(vlm_name, dtype=dtype, device_map="auto")
print(f"{vlm_name} cargado")

La función `preguntar` envía una imagen y una pregunta, y devuelve lo que contesta el modelo.

**Para que acierte:** una sola tarea por pregunta y diciendo cómo quieres la respuesta ("en una frase", "en JSON"). Entiende castellano, aunque en inglés acierta algo más.

In [ ]:
def preguntar(imagen, pregunta, max_new_tokens=200):
    """Envía una imagen y una pregunta al VLM y devuelve su respuesta."""
    mensajes = [{"role": "user", "content": [{"type": "image", "image": imagen}, {"type": "text", "text": pregunta}]}]
    entradas = vlm_processor.apply_chat_template(
        mensajes, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(vlm_model.device)
    with torch.no_grad():
        generado = vlm_model.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False)
    # La salida incluye la pregunta; nos quedamos solo con lo que ha escrito el modelo
    return vlm_processor.decode(generado[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

### Preguntar

Tres preguntas: qué pasa en la escena, una que obliga a razonar (en castellano) y una que obliga a leer el texto de la placa.

In [ ]:
mostrar(image_person_cars)
print(preguntar(image_person_cars, "What is the woman doing? Answer in one sentence."))
print(preguntar(image_person_cars, "¿Pueden arrancar ya los coches? Responde en una frase y explica por qué."))

mostrar(image_board)
print(preguntar(image_board, "What does the text on this board tell the user to do, and what do you think the board measures? Answer in one sentence."))

### Detectar con instrucciones

Si le pedimos las cajas en JSON, Qwen se convierte en un detector al que se le puede pedir casi cualquier cosa: "el bolso de la mujer", "el defecto de la tela", "el chip principal".

`detectar_con_vlm` añade a la instrucción el formato que queremos, lee el JSON de la respuesta y dibuja las cajas. Un detalle: Qwen3.5 da las coordenadas de 0 a 1000, no en píxeles, y la función las pasa al tamaño de la imagen.

In [ ]:
# Los "..." importan: si el ejemplo dijera "name", el modelo copiaría esa palabra como etiqueta.
FORMATO = ' Return only JSON in this format: [{"bbox_2d": [x1, y1, x2, y2], "label": "..."}]'


def detectar_con_vlm(imagen, instruccion):
    """Pide las cajas al VLM en JSON, las pasa a píxeles y las dibuja."""
    respuesta = preguntar(imagen, instruccion + FORMATO, max_new_tokens=512)
    print(respuesta)
    # El modelo suele envolver el JSON entre ```json y ```: se lo quitamos
    try:
        objetos = json.loads(respuesta.strip().removeprefix("```json").removesuffix("```"))
    except json.JSONDecodeError:
        objetos = []   # si no responde en JSON, no dibujamos nada

    ancho, alto = imagen.size
    cajas, etiquetas = [], []
    for objeto in objetos:
        x1, y1, x2, y2 = objeto["bbox_2d"]
        # De la escala 0-1000 a píxeles
        cajas.append([x1 * ancho / 1000, y1 * alto / 1000, x2 * ancho / 1000, y2 * alto / 1000])
        etiquetas.append(objeto["label"])
    dibujar_cajas(imagen, cajas, etiquetas, instruccion)


detectar_con_vlm(image_fruits, "Detect every fruit. Use the name of each fruit as its label.")


detectar_con_vlm(image_person_cars, "Detect the woman's bag.")
detectar_con_vlm(image_carpet_nok, "Detect the defect in the fabric.")
# detectar_con_vlm(image_board, "Detect the main chip and the blue potentiometer.")


### Pensar primero, detectar después

Pídele directamente "la fruta que le cayó a Newton en la cabeza" y puede marcar otra: tiene que deducir qué fruta es y localizarla a la vez. Si se lo preguntas por texto, responde "apple" sin dudar.

El truco es repartir el trabajo en dos llamadas: primero la pregunta, después la detección con la respuesta. Sirve con cualquier VLM y para cualquier tarea en la que haya que razonar antes de localizar.

In [ ]:
fruta = preguntar(image_fruits, "Which fruit fell on Newton's head? Answer with one word.")
print("El modelo dice:", fruta)
detectar_con_vlm(image_fruits, f"Detect the {fruta}.")

## Cuándo usar cada uno

| Modelo | Mejor para | A favor | En contra |
|---|---|---|---|
| **CLIP** | Clasificar con clases que eliges tú | Muy rápido y ligero | Solo clasifica la imagen entera |
| **BLIP** | Describir imágenes | Frases naturales | No localiza objetos |
| **Grounding DINO** | Detectar objetos nombrados | Cajas precisas | Entiende nombres, no instrucciones largas |
| **Qwen3.5** | Preguntas, leer texto, detectar con instrucciones | Un solo modelo para muchas tareas, entiende castellano | Más lento y pesado; conviene revisar sus respuestas |

Regla práctica: usa el modelo más sencillo que resuelva el problema. Si con CLIP basta, no hace falta un VLM.

## Extra: un prompt para cada modelo

Si vamos bien de tiempo lo hacemos juntos en clase; si no, queda para casa. Completa los huecos `___` y quita el `#` de las líneas de prueba.

Cada modelo espera el texto a su manera. Completa los prompts:

In [ ]:
# Hueco 1: la descripción que falta para que CLIP acierte con la foto del paso de cebra (en inglés)
# clasificar_con_clip(image_person_cars, ["___", "an empty beach", "a kitchen"])

# Hueco 2: Grounding DINO, solo las fresas. Recuerda el formato: en minúsculas y terminado en punto.
# detectar_por_texto(image_fruits, "___")

# Hueco 3: Qwen, los dos coches que esperan delante del paso de cebra: un taxi y un coche blanco
# detectar_con_vlm(image_person_cars, "Detect the ___ taxi and the ___ car.")

# Hueco 4: pregúntale a Qwen, en castellano, qué pide hacer el texto impreso en la placa
# print(preguntar(image_board, "___"))

## Para casa: más cosas con Qwen y CLIP

Tres ejemplos más, ya comprobados. Ejecútalos cuando quieras.

**Lo que sabe del mundo.** Aprendió con millones de imágenes y textos de internet, así que reconoce lugares conocidos sin que se lo enseñes.

In [ ]:
mostrar(image_park)
print(preguntar(image_park, "Which city is this, and what is the large green area? Answer in one sentence."))

**Un informe de inspección en JSON.** Con campos fijos, la respuesta ya tiene el formato que necesitaría un programa de control de calidad.

In [ ]:
print(preguntar(image_carpet_nok, 'You inspect fabric in a factory. '
                'Return only JSON: {"defect": true or false, "type": "...", "location": "..."}'))

**Buscar fotos escribiendo, con CLIP.** En vez de comparar una imagen con varios textos, se compara un texto con muchas imágenes y se ordenan de más a menos parecida. Así funcionan los buscadores de fotos que entienden "perro en la playa" sin que nadie haya etiquetado nada.

In [ ]:
nombres = ["cars.jpg", "person_cars.jpg", "person_dog.jpg", "dog.jpg", "dog2.jpg", "fruits.jpg", "fruits3.jpg",
           "bananas.jpg", "board.jpg", "carpet_ok.jpg", "park.jpg", "person_banana.jpg", "cut.jpg"]
coleccion = [cargar_imagen(nombre) for nombre in nombres]


def buscar(texto, cuantas=3):
    """Enseña las fotos de la colección que mejor encajan con el texto."""
    entradas = clip_processor(text=[texto], images=coleccion, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        parecido = clip_model(**entradas).logits_per_text[0].softmax(dim=0)
    mejores = parecido.topk(cuantas)

    figura, ejes = plt.subplots(1, cuantas, figsize=(15, 4))
    for eje, valor, indice in zip(ejes, mejores.values, mejores.indices):
        eje.imshow(coleccion[indice])
        eje.set_title(f"{nombres[indice]} ({valor:.0%})")
        eje.axis("off")
    plt.suptitle(texto)
    plt.tight_layout()
    plt.show()


buscar("a dog sleeping on a sofa")
buscar("an aerial view of a city park")
# buscar("an electronic circuit")
# buscar("a woman holding a banana")

## Para casa: con una imagen tuya

Elige una de las funciones (`clasificar_con_clip`, `describir`, `detectar_por_texto`, `preguntar` o `detectar_con_vlm`) y pruébala con una imagen tuya. Puedes cargarla con `cargar_imagen("https://...")` o subirla:

```python
from google.colab import files
subido = files.upload()
mi_imagen = Image.open(next(iter(subido))).convert("RGB")
```

Ideas: pregunta a Qwen algo que exija razonar ("¿qué falta en esta mesa para poder comer?"), compara lo que detecta Grounding DINO con lo que detecta Qwen con la misma frase, o busca una imagen en la que CLIP se equivoque.

In [ ]:
# Tu código aquí

## Resumen

- **CLIP** compara una imagen con textos: clasifica con las clases que inventes, sin entrenar.
- **BLIP** describe una imagen con una frase.
- **Grounding DINO** detecta lo que nombras, con el formato `"objeto. otro objeto."`.
- **Qwen3.5** responde preguntas, lee texto, razona y detecta siguiendo instrucciones; sus cajas van de 0 a 1000 y hay que pasarlas a píxeles.
- Todos se cargan desde Hugging Face con `from_pretrained`. Los tres primeros esperan el texto en inglés; Qwen entiende también castellano.

**Fin del día 1.** Mañana empezamos con el cuaderno 4, DINO: [abrir en Colab](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/4_DINO.ipynb).